In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
load_dotenv()

print('Imports ready.')

Imports ready.


Load only relevant documents

In [2]:
health_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf').load()
crop_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/crop_disease.pdf').load()
agri_html = BSHTMLLoader('../../04_data_ingestion_document_processing/data/agriculture.html', open_encoding='utf-8', bs_kwargs={'features': 'html.parser'}).load()
agri_txt = TextLoader('../../04_data_ingestion_document_processing/data/agriculture.txt', encoding='utf-8').load()

all_docs = health_pdf + crop_pdf + agri_html + agri_txt
print(f'Loaded {len(all_docs)} documents')

Loaded 37 documents


Split and add metadata

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

chunks = splitter.split_documents(all_docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = 'pdf' if file_name.endswith('.pdf') else 'html' if file_name.endswith('.html') else 'txt'
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'
    
print(f'Created {len(chunks)} clean chunks')

Created 185 clean chunks


Build in-memory vector store

In [5]:
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print('Clean vector store ready')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Clean vector store ready


Define query transformer

In [6]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

transform_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Rewrite the user question to be more specific and focused on the original domain (health or agriculture). Do NOT introduce unrelated topics.'),
    ('human', '{question}')
])
query_transformer = transform_prompt | llm | StrOutputParser()
print('Transformer ready')

Transformer ready


Test original vs transformed retrieval

In [8]:
raw_query = 'What are Nigeria communicable and infectious diseases?'

# Original
orig_docs = retriever.invoke(raw_query)
print('Original query results:')
for d in orig_docs:
    print('-', d.page_content[:200])
    
    
# Transformed
trnasformed_query = query_transformer.invoke({'question': raw_query})
print(f'\nTransformed query: {trnasformed_query}')

trans_docs = retriever.invoke(trnasformed_query)
print('\nTransformed query results:')
for d in trans_docs:
    print(f'- {d.page_content}')

Original query results:
- scientific database sources, web search engines, direct observation and relevant documents from the Nigerian Ministry 
of Health. The major public health challenges Nigeria faces are infectious diseas
- Introduction Practice Points 
 Nigeria is often referred to as the "Giant of  
Africa", owing to its large population and  
economy, with approximately 182 million   
inhabitants.  
 Communicable an
- health workforce; medical products, vaccines and 
technologies; information; financing; and services 
delivery.17 
 
In Nigeria communicable and infectious diseases are 
the major health problem. 3 Ni
- programs designed to address each of these problems. The 
first WHO Global Status Report on non -communicable 
disease listed Nigeria and other developing countries as 
the worst hit with deaths from 

Transformed query: What are the most prevalent communicable and infectious diseases in Nigeria, and what measures are being taken to control their spread?

Transfo